# 9) Maths-2 Linear programming — geometry and duality — Exercises

**Goals:** enumerate the vertices of a feasible region, test the fundamental theorem of LP, measure shadow prices by finite differences (including one-sided ones at a kink), derive prices from the equal-marginal condition, trace how electricity and carbon prices respond to the CO₂ cap, and compare a carbon **tax** with a carbon **cap**.

The setup cell holds the lecture's two-plant problem. Costs in €/MWh, emissions in t/MWh, capacities in MW.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import itertools

import numpy as np
import pandas as pd
from scipy.optimize import linprog

COST = np.array([60.0, 90.0])      # coal, gas: EUR/MWh
EMIS = np.array([0.90, 0.35])      # t/MWh
CAP = np.array([600.0, 800.0])     # MW
DEMAND, CO2_CAP = 1000.0, 600.0


def dispatch(demand=DEMAND, co2_cap=CO2_CAP, cost=COST):
    """The lecture's LP (CO2 cap optional: pass co2_cap=None)."""
    kwargs = {} if co2_cap is None else {"A_ub": [EMIS], "b_ub": [co2_cap]}
    return linprog(cost, A_eq=[[1.0, 1.0]], b_eq=[demand], bounds=list(zip([0.0, 0.0], CAP)),
                   method="highs", **kwargs)

### Warm-ups

**1. Vertices of a polygon**

Write `vertices(A, b)` returning an array of all vertices of {x : A x ≤ b} in two dimensions (intersect every pair of constraint lines, keep the feasible points, remove duplicates).

In [2]:
def vertices(A, b, tol=1e-9) -> np.ndarray:
    """Vertices (rows) of the 2-D polygon A x <= b."""
    ### BEGIN SOLUTION
    A, b = np.asarray(A, float), np.asarray(b, float)
    points = []
    for i, j in itertools.combinations(range(len(A)), 2):
        M = A[[i, j]]
        if abs(np.linalg.det(M)) < tol:
            continue
        x = np.linalg.solve(M, b[[i, j]])
        if np.all(A @ x <= b + 1e-7):
            points.append(np.round(x, 9))
    return np.unique(np.array(points), axis=0)
    ### END SOLUTION

In [3]:
square = vertices([[1, 0], [-1, 0], [0, 1], [0, -1]], [1, 0, 1, 0])
assert len(square) == 4 and {tuple(v) for v in square} == {(0, 0), (0, 1), (1, 0), (1, 1)}
A_lec = [[-1, 0], [0, -1], [1, 0], [0, 1], [0.90, 0.35], [-1, -1]]
b_lec = [0, 0, 600, 800, CO2_CAP, -DEMAND]
assert len(vertices(A_lec, b_lec)) == 3, "the lecture's triangle"

**2. The fundamental theorem, tested**

For 200 random cost vectors (positive or negative), solve min cᵀx over the lecture's polygon with `linprog` and check that the solution is always one of the vertices — and that it has the lowest cost among them.

In [4]:
def optimum_is_a_vertex(n_trials: int = 200, seed: int = 0) -> bool:
    """True if every random-cost LP over the lecture's polygon is solved at its cheapest vertex."""
    ### BEGIN SOLUTION
    rng = np.random.default_rng(seed)
    V = vertices(A_lec, b_lec)
    for _ in range(n_trials):
        c = rng.normal(size=2)
        x = linprog(c, A_ub=A_lec, b_ub=b_lec, bounds=[(None, None)] * 2, method="highs").x
        if not np.isclose(c @ x, (V @ c).min(), rtol=1e-9, atol=1e-6):
            return False
    return True
    ### END SOLUTION

In [5]:
assert optimum_is_a_vertex(), "an LP optimum is always attained at a vertex"

### Core

**3. One-sided shadow prices**

Write `one_sided_prices(demand, co2_cap=None)` returning the cost of the **last** MWh (cost(D) − cost(D − 1)) and of the **next** MWh (cost(D + 1) − cost(D)). Away from a kink they agree; at a kink (lecture: D = 600 MW without cap) they do not.

In [6]:
def one_sided_prices(demand: float, co2_cap=None) -> tuple[float, float]:
    """(price of the last MWh, price of the next MWh) in EUR/MWh."""
    ### BEGIN SOLUTION
    f = lambda d: dispatch(d, co2_cap).fun
    return f(demand) - f(demand - 1), f(demand + 1) - f(demand)
    ### END SOLUTION

In [7]:
assert np.allclose(one_sided_prices(400), (60, 60)), "only coal at the margin"
assert np.allclose(one_sided_prices(600), (60, 90)), "a kink: degenerate price"
left, right = one_sided_prices(1000, CO2_CAP)
assert np.isclose(left, right) and np.isclose(left, dispatch().eqlin.marginals[0]), "with the cap: 109.09 both ways"

**4. Prices from the equal-marginal condition**

When both plants run strictly between 0 and capacity and the cap binds, λ = c_i + μ·e_i for both. Write `equal_marginal_prices(cost, emis)` returning (λ, μ), and check it against the solver for caps of 600 and 650 t. What do you notice?

In [8]:
def equal_marginal_prices(cost, emis) -> tuple[float, float]:
    """Electricity price (EUR/MWh) and carbon price (EUR/t) when both plants are marginal."""
    ### BEGIN SOLUTION
    mu = (cost[1] - cost[0]) / (emis[0] - emis[1])
    return cost[0] + mu * emis[0], mu
    ### END SOLUTION

In [9]:
lam, mu = equal_marginal_prices(COST, EMIS)
for cap_t in [600, 650]:
    r = dispatch(co2_cap=cap_t)
    assert np.isclose(r.eqlin.marginals[0], lam) and np.isclose(-r.ineqlin.marginals[0], mu)
print(f"lambda = {lam:.2f} EUR/MWh, mu = {mu:.2f} EUR/t, for both caps")

lambda = 109.09 EUR/MWh, mu = 54.55 EUR/t, for both caps


**5. How the prices respond to the cap**

Write `cap_sweep(caps)` returning a DataFrame (index: cap in t) with the dispatch of both plants, the electricity price and the carbon price (positive, €/t) — and `None`/NaN rows when the cap cannot be met. Sweep caps from 400 to 720 t.

In [10]:
def cap_sweep(caps) -> pd.DataFrame:
    """Dispatch and prices for each CO2 cap (NaN when infeasible)."""
    ### BEGIN SOLUTION
    rows = {}
    for cap_t in caps:
        r = dispatch(co2_cap=cap_t)
        if r.status != 0:
            rows[cap_t] = dict(coal_MW=np.nan, gas_MW=np.nan, price_EUR_MWh=np.nan, carbon_EUR_t=np.nan)
        else:
            rows[cap_t] = dict(coal_MW=r.x[0], gas_MW=r.x[1], price_EUR_MWh=r.eqlin.marginals[0],
                               carbon_EUR_t=-r.ineqlin.marginals[0])
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("co2_cap_t")
    ### END SOLUTION

In [11]:
sweep = cap_sweep(range(400, 721, 40))
assert sweep.loc[400].isna().all(), "below 460 t even maximum gas cannot meet demand"
assert np.isclose(sweep.loc[720, "carbon_EUR_t"], 0) and np.isclose(sweep.loc[720, "price_EUR_MWh"], 90), "cap not binding"
feasible = sweep.dropna()
assert feasible.carbon_EUR_t.is_monotonic_decreasing and feasible.coal_MW.is_monotonic_increasing
sweep.round(2)

           coal_MW  gas_MW  price_EUR_MWh  carbon_EUR_t
co2_cap_t                                              
400            NaN     NaN            NaN           NaN
440            NaN     NaN            NaN           NaN
480         236.36  763.64         109.09         54.55
520         309.09  690.91         109.09         54.55
560         381.82  618.18         109.09         54.55
600         454.55  545.45         109.09         54.55
640         527.27  472.73         109.09         54.55
680         600.00  400.00          90.00          0.00
720         600.00  400.00          90.00          0.00

**6. Interpretation**

Your sweep shows the carbon and electricity prices as *steps*, not smooth curves. Explain the three regimes (infeasible, cap binding, cap slack) and why the prices stay constant within the middle regime even though the dispatch changes. What would make the curves smoother in a real system?

<!-- BEGIN SOLUTION -->
Below 460 t the cap cannot be met at all: even with gas at its 800 MW limit (280 t), the remaining 200 MW of coal add 180 t, so emissions cannot fall below 460 t. Between 460 and 680 t the cap binds and **both** plants run between their limits, so the equal-marginal condition fixes λ = 109.09 €/MWh and μ = 54.55 €/t regardless of the cap's exact level — tightening the cap only moves the dispatch along the demand line (less coal, more gas), at constant marginal trade-off. Above 680 t the cap no longer binds (coal already runs at capacity), so μ = 0 and the price falls to the marginal plant's cost, 90 €/MWh. Prices are steps because an LP's duals change only when the set of binding constraints changes (a new vertex). A real system has many plants with different costs and emission factors, so each tightening of the cap brings in a new pair of marginal plants with its own μ: the steps become many and small, and the curves look smooth — this is the marginal abatement cost curve behind an emissions-trading price.
<!-- END SOLUTION -->

### Challenge

**7. A carbon tax instead of a cap**

Remove the cap and add a carbon **tax** τ (€/t) to each plant's cost: c_i + τ·e_i. Write `emissions_with_tax(tau)` returning the resulting emissions (t). Show that a tax just below the cap's shadow price μ and one just above it give very different emissions, and find the tax at which the switch happens by bisection.

In [12]:
def emissions_with_tax(tau: float) -> float:
    """Emissions (t) of the cost-minimal dispatch under a carbon tax tau (EUR/t), no cap."""
    ### BEGIN SOLUTION
    r = dispatch(co2_cap=None, cost=COST + tau * EMIS)
    return float(EMIS @ r.x)
    ### END SOLUTION


def switching_tax(lo: float = 0.0, hi: float = 200.0, tol: float = 1e-6) -> float:
    """Tax (EUR/t) at which the dispatch switches from coal-heavy to gas-heavy (bisection)."""
    ### BEGIN SOLUTION
    high_emissions = emissions_with_tax(lo)
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if np.isclose(emissions_with_tax(mid), high_emissions):
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2
    ### END SOLUTION

In [13]:
assert np.isclose(emissions_with_tax(54), 680), "just below mu: coal runs flat out"
assert np.isclose(emissions_with_tax(55), 460), "just above mu: gas runs flat out"
tau_star = switching_tax()
assert np.isclose(tau_star, mu, atol=1e-4), "the switching tax equals the cap's shadow price"
print(f"emissions jump from 680 t to 460 t at a tax of {tau_star:.4f} EUR/t (cap shadow price {mu:.4f} EUR/t)")

emissions jump from 680 t to 460 t at a tax of 54.5455 EUR/t (cap shadow price 54.5455 EUR/t)


**8. Interpretation**

The cap delivered exactly 600 t at a carbon price of 54.55 €/t. Why can no tax in this two-plant model deliver exactly 600 t? What does that tell you about the difference between price instruments (taxes) and quantity instruments (caps, like the EU ETS), and why does the difference shrink in a real power system?

<!-- BEGIN SOLUTION -->
With a tax the problem has no cap, so each plant's taxed cost decides the merit order. Below τ = 54.55 €/t coal is cheaper (including its carbon) and runs at capacity (680 t); above it gas is cheaper and runs at capacity (460 t). At exactly 54.55 €/t both cost 109.09 €/MWh and *every* split is optimal — a degenerate LP — so 600 t is only one of infinitely many solutions, and nothing in the tax picks it. A cap fixes the **quantity** and lets the price adjust (the price emerges as the dual μ); a tax fixes the **price** and lets the quantity adjust. In theory they are equivalent when the tax equals the cap's shadow price, but with few, lumpy options the quantity response to a price is a step, so a tax cannot hit a target precisely and a cap's price can jump. The EU ETS is a quantity instrument for this reason: emissions are certain, the price is not. In a real system with many plants of different costs and emission factors, small changes in the carbon price switch only a few plants, the emissions response becomes nearly continuous, and the two instruments converge.
<!-- END SOLUTION -->